In [2]:
import json

with open("school_codes_list.json", "r") as f:
    school_codes_list = json.load(f)

In [1]:
import pandas as pd
import requests
from io import StringIO

url = "https://www3.cde.ca.gov/demo-downloads/staff/strat2425.txt"

r = requests.get(url)
r.raise_for_status()
text = r.content.decode("cp1252", errors="replace")

In [6]:
df = pd.read_csv(
    StringIO(text),
    sep="\t",
    dtype={
        "Academic Year": "string",
        "Aggregate Level": "string",
        "County Code": "string",
        "District Code": "string",
        "School Code": "string",
        "County Name": "string",
        "District Name": "string",
        "School Name": "string",
        "Charter School": "string",
        "DASS": "string",
        "School Grade Span": "string"
    },
    keep_default_na=False
)

df.columns = df.columns.str.strip()

string_cols = [
    "Academic Year", "Aggregate Level", "County Code", "District Code", "School Code",
    "County Name", "District Name", "School Name", "Charter School", "DASS",
    "School Grade Span"
]

for col in string_cols:
    df[col] = df[col].str.strip()

# keep only school-level rows
staff_df = df[df["Aggregate Level"] == "S"].copy()

# fix code formatting
staff_df["School Code"] = staff_df["School Code"].astype(int).astype(str).str.zfill(7)

# filter to your schools
staff_df = staff_df[staff_df["School Code"].isin(school_codes_list)].copy()
staff_df = staff_df.reset_index(drop=True)

In [8]:
staff_df = staff_df[['School Code', 'STU_TCH_RATIO']]

In [9]:
staff_df

,School Code,STU_TCH_RATIO
0,0130625,22.1
1,0106401,27.4
2,0130229,23.8
3,0130450,18.8
4,0131177,18.1
...,...,...
1392,5738802,19
1393,5830047,11
1394,5830013,21
1395,5835202,20


In [ ]:
staff_df.to_csv('../cleaned_data/teacher_ratio_data.csv')